# 📊 Data Lead Pipeline — Arabic + English Invoice Dataset (Kaggle Edition)

**Role:** Data Lead (Person 1)  
**Mission:** Build, validate, stratify, and deliver the benchmark dataset for the entire 6-person team.

---
### ⚡ What this Kaggle Pipeline does:
1. **Environment Setup:** Installs dependencies and headless Chromium browser.
2. **Public Benchmarks:** Downloads and standardizes **SROIE** & **CORD-v2** (~2,000 receipts).
3. **Synthetic Generator:** Generates **~3,500 Arabic & Mixed Invoices** with 100% correct ground truth.
4. **Quality & Schema Validation:** Enforces strict 0-error schema.
5. **Stratified Splitting & Leakage Verification:** Creates **Train (80%), Val (15%), Test (5%)** with 0 duplicate images.
6. **EDA Report:** Statistical tables and distributions.
7. **One-Click Export:** Archives everything into a ready-to-share `dataset_v1.zip` in Kaggle Cloud.

## 1️⃣ Environment Setup & Dependencies Installation
> ⚠️ **Important:** Make sure **Internet: ON** is enabled in the right sidebar of your Kaggle Notebook!

In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path

# Ensure working directory is set to /kaggle/working
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')
print(f"📁 Current Working Directory: {os.getcwd()}")

# Auto-detect scripts if uploaded as a Kaggle Dataset / Input
input_dir = Path('/kaggle/input')
if input_dir.exists() and not Path('scripts').exists():
    for p in input_dir.rglob('kaggle_upload.zip'):
        with zipfile.ZipFile(p, 'r') as z:
            z.extractall('.')
        print(f"✅ Extracted scripts from {p}")
        break
    if not Path('scripts').exists():
        for p in input_dir.rglob('scripts'):
            if p.is_dir():
                shutil.copytree(p, 'scripts', dirs_exist_ok=True)
                print(f"✅ Copied scripts from {p}")
                break

# Install required Python libraries
!pip install -q datasets huggingface_hub arabic-reshaper python-bidi pillow matplotlib pandas tqdm opencv-python

# Install Playwright and headless Chromium browser for high-speed rendering
!pip install -q playwright && playwright install --with-deps chromium

print("\n✅ Environment ready and Chromium installed successfully!")

## 2️⃣ Download and Convert Real Benchmark Datasets (SROIE & CORD)

In [ ]:
# 1. Download SROIE dataset using huggingface-cli to download binary images
!mkdir -p data/raw
if not os.path.exists('data/raw/sroie/train') and not os.path.exists('data/raw/sroie/task3'):
    !huggingface-cli download --repo-type dataset jsdnrs/ICDAR2019-SROIE --local-dir data/raw/sroie --quiet

# 2. Convert SROIE
!python scripts/convert_sroie.py

# 3. Convert CORD-v2 from Hugging Face
!python scripts/convert_cord.py

## 3️⃣ High-Speed Synthetic Generation (Arabic RTL + Mixed Invoices)
Generates realistic synthetic invoices with ground truth targeting 4 core fields:
- `supplier`
- `invoice_number`
- `date`
- `total`

In [ ]:
# Generate 3,500 balanced synthetic invoices (55% Arabic, 35% Mixed, 10% English)
!python scripts/generate_synthetic.py --count 3500

## 4️⃣ Validate Dataset (Schema & Quality Control)
Verifies that all records match the data contract with 0 errors.

In [ ]:
!python scripts/validate_dataset.py

## 5️⃣ Stratified Train / Val / Test Splits (80% / 15% / 5%) & Leakage Check
Partitions the dataset into 80% Train, 15% Validation, and 5% Test, and cryptographic MD5 checks confirm 0 duplicates.

In [ ]:
# 1. Create stratified splits (80% Train, 15% Val, 5% Test)
!python scripts/create_splits.py --train 0.80 --val 0.15 --test 0.05

# 2. Verify 0 leakage between splits
!python scripts/check_leakage.py

## 6️⃣ Exploratory Data Analysis (EDA) Report

In [ ]:
!python scripts/run_eda.py

## 7️⃣ Summary & Archive Creation in Kaggle Cloud
Packages `dataset_v1.zip` directly in your Kaggle Cloud storage.

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("🎉 DATA LEAD DELIVERY AUDIT (80/15/5 SPLITS)")
print("=" * 60)

for s in ["train", "val", "test"]:
    p = Path(f"data/splits/{s}.jsonl")
    if p.exists():
        count = sum(1 for _ in open(p, encoding="utf-8"))
        print(f"  ✅ Split {s.upper():<5} : {count:,} records")
    else:
        print(f"  ❌ Split {s.upper():<5} : NOT FOUND!")

print("\n📦 Compressing deliverable into dataset_v1.zip in Kaggle Cloud...")
!zip -q -r /kaggle/working/dataset_v1.zip data/splits data/processed/images data/synth/out/images docs/eda_report.md

if os.path.exists('/kaggle/working/dataset_v1.zip'):
    size_mb = os.path.getsize('/kaggle/working/dataset_v1.zip') / (1024 * 1024)
    print(f"\n🚀 SUCCESS! Archive created: dataset_v1.zip ({size_mb:.1f} MB)")
    print("✨ The dataset is saved entirely in the Cloud without downloading to your PC.")
print("=" * 60)